[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [SQLAlchemy, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlalchemy-deep-dive.html)

# Async SQLAlchemy


## What you will be able to do

Make an async engine and `AsyncSession`s with `create_async_engine` and `async_sessionmaker`, await
every call that reaches the database, and read results all at once or streamed. Load relationships
in the query or through `awaitable_attrs`, run synchronous ORM code with `run_sync`, and run several
sessions at once with `asyncio.gather`. Recognize the `MissingGreenlet` error from an attribute that
needed the database, a call whose `await` was left out, `asyncio.run()` inside a notebook, and one
session shared between tasks.


## The idea

### The problem

A web server answers many requests at once, and a request spends most of its time waiting: for the
database, for a file, for another service. Python's `asyncio` runs many waits on one thread, but only
for code that says where it waits, with `await`. An ordinary database call in the middle of that code
holds the thread for as long as the database takes, and every other request on it waits too.

SQLAlchemy's asyncio extension gives the engine and the session calls that can be awaited:
`await session.execute(...)` hands the thread back while the database works. Most of the ORM carries
over unchanged. The part that does not is the part that happens without being asked: a lazy load is
a query hidden in an attribute read, and an attribute read cannot be awaited. In an async session,
the attribute that was never loaded raises an error about greenlets, which the code never mentioned.

This notebook runs on SQLite, which a program reaches as a file on the same machine, so it shows how
async code is written, not how much sooner it finishes. A timing would print a different number on
every machine, and the notebook prints none.

### What async SQLAlchemy is

> **`create_async_engine()`** makes an engine for an async driver, here `sqlite+aiosqlite`, and
> **`async_sessionmaker()`** makes **`AsyncSession`**s from it. Every session method that talks to
> the database is a coroutine, to be awaited: `execute`, `scalars`, `scalar`, `get`, `commit`,
> `refresh`, `delete`. `add()` only hands an object to the session, and is not awaited.
> **`selectinload`** and the other eager loads fetch relationships inside the awaited query,
> **`awaitable_attrs`**, from the **`AsyncAttrs`** mixin, loads one attribute when it is awaited, and
> **`run_sync()`** runs ordinary synchronous code, lazy loads included, inside the async session.
> **`MissingGreenlet`** is the error an attribute raises when reading it would need the database.

### Why it works that way

- **An attribute read cannot wait.** `student.enrollments` is not a call, so nothing can await it,
  and a lazy load inside it would hold the thread. SQLAlchemy raises instead.
- **The async ORM is the ordinary ORM, run on a greenlet.** The extension runs the ORM's own
  synchronous code inside a greenlet that can pause at every database call and hand control back to
  the event loop. Code outside that greenlet that reaches the database has nowhere to pause, which is
  what "greenlet_spawn has not been called" means.
- **A commit expires what it saved, by default.** Reading an expired attribute is a lazy load too, so
  async sessions are usually made with `expire_on_commit=False`.
- **A notebook already runs an event loop.** A cell can `await` at its top level, and
  `asyncio.run()`, which starts a loop of its own, refuses to run inside one. A script has no loop
  until `asyncio.run()` starts it.
- **One session is one conversation.** `asyncio.gather` runs several coroutines at once, and each
  needs a session of its own, since a session sends one statement at a time.

### Where this shows up

FastAPI, which the **APIs and JSON** guide used for its servers, accepts endpoints written with
`async def`, and an endpoint written that way needs database calls it can await. The
**Loading Strategies** notebook's `selectinload` and `raiseload` are how an async program decides
ahead of time what it will read, and **The Identity Map** notebook's `expire_on_commit` is the
setting async sessions change.

### What this notebook covers

- An async engine for the same college
- Awaiting every call that reaches the database
- Results all at once, or streamed
- Relationships: loaded in the query, or awaited
- Synchronous code in an async session: `run_sync`
- Several sessions at once: `asyncio.gather`
- Which to use, sync or async
- Transcripts for an advising page, finished
- Four errors, from an attribute that needed the database to one session shared between tasks

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read
it, and read the output underneath it. Everything from Setup onward is where you start running
things, and the rest of the notebook takes this apart piece by piece.

```python
import asyncio

from sqlalchemy import ForeignKey, select
from sqlalchemy.ext.asyncio import async_sessionmaker, create_async_engine
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column, relationship, selectinload


class Base(DeclarativeBase):
    pass


class Course(Base):
    __tablename__ = "courses"
    id: Mapped[int] = mapped_column(primary_key=True)
    code: Mapped[str]
    sections: Mapped[list["Section"]] = relationship()


class Section(Base):
    __tablename__ = "sections"
    id: Mapped[int] = mapped_column(primary_key=True)
    course_id: Mapped[int] = mapped_column(ForeignKey("courses.id"))


async def main():
    engine = create_async_engine("sqlite+aiosqlite://")
    async with engine.begin() as conn:
        await conn.run_sync(Base.metadata.create_all)
    Session = async_sessionmaker(engine, expire_on_commit=False)
    async with Session() as session:
        session.add_all([Course(code="BIO-101", sections=[Section(), Section()]),
                         Course(code="STA-200", sections=[Section()])])
        await session.commit()
    async with Session() as session:
        with_sections = select(Course).options(selectinload(Course.sections))
        courses = (await session.scalars(with_sections)).all()
        print([(course.code, len(course.sections)) for course in courses])
    async with Session() as session:
        courses = (await session.scalars(select(Course))).all()
        try:
            print(len(courses[0].sections))
        except Exception as error:
            print(type(error).__name__)
    await engine.dispose()


asyncio.run(main())
```

```
[('BIO-101', 2), ('STA-200', 1)]
MissingGreenlet
```

Every call that reached the database was awaited, and `asyncio.run()` started the event loop that
ran them, as a script must. The query with `selectinload` brought the sections with the courses, so
`len(course.sections)` needed nothing more. The query without it left `sections` unloaded, and
reading it would have needed a query that nothing could await, so it raised `MissingGreenlet`.


## Setup

Ten imports, and the college built from its classes.

- `sqlalchemy` is the library itself, and the cell prints its version
- `create_async_engine` and `async_sessionmaker`, from `sqlalchemy.ext.asyncio`, make the async
  engine and its sessions, and `AsyncAttrs` gives every object `awaitable_attrs`
- `asyncio` runs several coroutines at once with `gather`, and is also what a notebook cannot call
  `run()` on, in Common errors
- `selectinload`, from `sqlalchemy.orm`, loads relationships inside a query, with `relationship` and
  the rest of what the classes need
- `select`, `func` and `text` write the queries, and `insert`, `create_engine` and `event` build the
  college and its engines
- `StaticPool`, from `sqlalchemy.pool`, is the pool the helper uses for a database in memory
- `date` is what the `Date` columns take and return
- `logging` carries the SQL an engine logs to `PrintStatements`
- `Path` names the files, and `shutil` removes the scratch folder at the start and at the end

Setup builds the college with the ordinary engine, as every notebook from **Many to Many** on does,
from the classes of the **Relationships** notebook. The classes differ in one line: `Base` also
inherits `AsyncAttrs`, which the fourth worked example uses. The async engine comes in the first
worked example.

Colab has SQLAlchemy, `aiosqlite` and `greenlet` installed, and this notebook runs SQLAlchemy 2.0.54
with `aiosqlite` 0.22.1. Any 2.0 release runs it, though an error may be worded a little
differently. To match it exactly, run `%pip install sqlalchemy==2.0.54` in a cell of its own, restart
the session, and run this cell again.


In [1]:
import asyncio
import logging
import shutil
from datetime import date
from pathlib import Path

import sqlalchemy
from sqlalchemy import (CheckConstraint, ForeignKey, MetaData, String, UniqueConstraint, create_engine, event, func, insert,
                        select, text)
from sqlalchemy.ext.asyncio import AsyncAttrs, async_sessionmaker, create_async_engine
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column, relationship, selectinload
from sqlalchemy.pool import StaticPool

SCRATCH = Path("scratch")
shutil.rmtree(SCRATCH, ignore_errors=True)
SCRATCH.mkdir()
DATABASE = SCRATCH / "college.db"

NAMES = [
    "Ana Reyes", "Ben Okafor", "Chloe Martin", "Daniel Kim", "Elena Petrova", "Felix Wagner",
    "Grace Lin", "Hassan Ali", "Isabel Costa", "Jonas Berg", "Keiko Tanaka", "Liam Murphy",
    "Maya Patel", "Noah Andersen", "Olivia Brandt", "Pavel Novak", "Quinn Harper", "Rosa Delgado",
    "Sam Ito", "Tara Nilsen", "Umar Farouk", "Vera Kowalski", "Wes Carter", "Yara Haddad",
    "Aoife O'Brien",
]
PROGRAMS = ["Biology", "Computer Science", "Mathematics", "Psychology", "History"]
TERMS = [("Fall 2024", "2024-08-26"), ("Spring 2025", "2025-01-13"), ("Fall 2025", "2025-08-25"),
         ("Spring 2026", "2026-01-12")]
STUDENTS = [(name, f"{name[0]}{name.split()[-1]}@college.edu".lower().replace("'", ""),
             PROGRAMS[i % len(PROGRAMS)], TERMS[i % 3][1]) for i, name in enumerate(NAMES)]
COURSES = [
    ("BIO-101", "Introduction to Biology", "Biology", 4),
    ("CHE-110", "General Chemistry", "Chemistry", 4),
    ("MAT-120", "Calculus I", "Mathematics", 4),
    ("MAT-121", "Calculus II", "Mathematics", 4),
    ("CSC-101", "Programming I", "Computer Science", 3),
    ("CSC-201", "Data Structures", "Computer Science", 3),
    ("ENG-105", "Composition", "English", 3),
    ("HIS-110", "World History", "History", 3),
    ("PSY-101", "Introduction to Psychology", "Psychology", 3),
    ("STA-200", "Statistics", "Mathematics", 3),
]
GRADES = ["A", "A-", "B+", "B", "B-", "C+", "C", "C-", "D", "F"]

# One section of every course in every term, so the section of course c in term t has id (t - 1) * 10 + c.
SECTIONS = [(course, term, 30) for term in range(1, len(TERMS) + 1) for course in range(1, len(COURSES) + 1)]

# Three courses a term for every student, from the term they started. Spring 2026 is under way.
ENROLLMENTS = []
for s in range(len(NAMES)):
    for term in range(s % 3 + 1, len(TERMS) + 1):
        for k in range(3):
            section = (term - 1) * len(COURSES) + (s + term + 3 * k) % len(COURSES) + 1
            if term < len(TERMS):
                ENROLLMENTS.append((s + 1, section, "completed", GRADES[(s * 7 + term * 5 + k * 3) % len(GRADES)]))
            else:
                ENROLLMENTS.append((s + 1, section, "enrolled", None))

class PrintStatements(logging.Handler):
    """Print what an engine logs, leaving out the time: every statement, and the values sent with it."""

    def emit(self, record):
        if record.msg == "[%s] %r":                  # after a statement: how long it took, then its values
            values = repr(record.args[1])
            if values != "()":
                print("    values:", values)
        else:
            for line in record.getMessage().splitlines():
                print("   ", line.rstrip())


sql_log = logging.getLogger("sqlalchemy.engine.Engine")
sql_log.handlers = [PrintStatements()]              # this handler alone, however often the cell runs
sql_log.propagate = False                           # and no handler above it prints the same lines again


def college_engine(path=None, echo=False):
    """An engine for the college's database, in a file or in memory, with foreign keys enforced."""
    if path is None:                                # in memory: one connection, and one database, for every thread
        engine = create_engine("sqlite://", poolclass=StaticPool, echo=echo,
                               connect_args={"check_same_thread": False, "autocommit": False})
    else:
        engine = create_engine(f"sqlite:///{path}", echo=echo, connect_args={"autocommit": False})

    @event.listens_for(engine, "connect")
    def enforce_foreign_keys(dbapi_connection, connection_record):
        dbapi_connection.autocommit = True          # the PRAGMA does nothing inside a transaction,
        dbapi_connection.execute("PRAGMA foreign_keys = ON")
        dbapi_connection.autocommit = False         # and with autocommit=False sqlite3 keeps one open

    return engine

NAMING = {
    "pk": "pk_%(table_name)s",
    "uq": "uq_%(table_name)s_%(column_0_N_name)s",
    "ck": "ck_%(table_name)s_%(constraint_name)s",
    "fk": "fk_%(table_name)s_%(column_0_name)s_%(referred_table_name)s",
    "ix": "ix_%(column_0_label)s",
}


GRADE_POINTS = {"A": 4.0, "A-": 3.7, "B+": 3.3, "B": 3.0, "B-": 2.7, "C+": 2.3, "C": 2.0, "C-": 1.7, "D": 1.0, "F": 0.0}


class Base(AsyncAttrs, DeclarativeBase):
    metadata = MetaData(naming_convention=NAMING)


class Student(Base):
    __tablename__ = "students"

    id: Mapped[int] = mapped_column(primary_key=True)
    name: Mapped[str] = mapped_column(String(100))
    email: Mapped[str] = mapped_column(String(200), unique=True)
    program: Mapped[str] = mapped_column(String(50))
    started_on: Mapped[date]

    enrollments: Mapped[list["Enrollment"]] = relationship(back_populates="student", order_by="Enrollment.section_id")

    def __repr__(self):
        return f"Student({self.name!r}, {self.program!r})"


class Course(Base):
    __tablename__ = "courses"
    __table_args__ = (CheckConstraint("credits BETWEEN 1 AND 6", name="credits_range"),)

    id: Mapped[int] = mapped_column(primary_key=True)
    code: Mapped[str] = mapped_column(String(10), unique=True)
    title: Mapped[str] = mapped_column(String(100))
    department: Mapped[str] = mapped_column(String(50))
    credits: Mapped[int]

    sections: Mapped[list["Section"]] = relationship(back_populates="course", order_by="Section.term_id")

    def __repr__(self):
        return f"Course({self.code!r}, {self.credits})"


class Term(Base):
    __tablename__ = "terms"

    id: Mapped[int] = mapped_column(primary_key=True)
    name: Mapped[str] = mapped_column(String(20), unique=True)
    starts_on: Mapped[date]

    sections: Mapped[list["Section"]] = relationship(back_populates="term", order_by="Section.course_id")

    def __repr__(self):
        return f"Term({self.name!r})"


class Section(Base):
    __tablename__ = "sections"
    __table_args__ = (UniqueConstraint("course_id", "term_id"), CheckConstraint("capacity > 0", name="capacity_positive"))

    id: Mapped[int] = mapped_column(primary_key=True)
    course_id: Mapped[int] = mapped_column(ForeignKey("courses.id"))
    term_id: Mapped[int] = mapped_column(ForeignKey("terms.id"))
    capacity: Mapped[int]

    course: Mapped["Course"] = relationship(back_populates="sections")
    term: Mapped["Term"] = relationship(back_populates="sections")
    enrollments: Mapped[list["Enrollment"]] = relationship(back_populates="section", order_by="Enrollment.student_id")

    def __repr__(self):
        return f"Section({self.id})"


class Enrollment(Base):
    __tablename__ = "enrollments"
    __table_args__ = (CheckConstraint("status IN ('enrolled', 'completed', 'withdrawn')", name="status_known"),)

    student_id: Mapped[int] = mapped_column(ForeignKey("students.id"), primary_key=True)
    section_id: Mapped[int] = mapped_column(ForeignKey("sections.id"), primary_key=True)
    status: Mapped[str] = mapped_column(String(20), server_default="enrolled")
    grade: Mapped[str | None] = mapped_column(String(2))

    student: Mapped["Student"] = relationship(back_populates="enrollments")
    section: Mapped["Section"] = relationship(back_populates="enrollments")

    @property
    def grade_points(self):
        """The points the grade is worth, or None before there is a grade."""
        return None if self.grade is None else GRADE_POINTS[self.grade]

    def __repr__(self):
        return f"Enrollment(student {self.student_id}, section {self.section_id}, {self.grade!r})"


def build_college(engine):
    """Create the college's tables from the classes, load the lists above into them, and count their rows."""
    Base.metadata.create_all(engine)
    rows = {
        Course: [{"code": code, "title": title, "department": department, "credits": credits}
                 for code, title, department, credits in COURSES],
        Student: [{"name": name, "email": email, "program": program, "started_on": date.fromisoformat(started)}
                  for name, email, program, started in STUDENTS],
        Term: [{"name": name, "starts_on": date.fromisoformat(starts)} for name, starts in TERMS],
        Section: [{"course_id": course, "term_id": term, "capacity": capacity} for course, term, capacity in SECTIONS],
        Enrollment: [{"student_id": student, "section_id": section, "status": status, "grade": grade}
                     for student, section, status, grade in ENROLLMENTS],
    }
    with engine.begin() as conn:
        for cls, values in rows.items():
            conn.execute(insert(cls), values)
        return {cls.__tablename__: conn.execute(select(func.count()).select_from(cls)).scalar_one() for cls in rows}


engine = college_engine(DATABASE)
print("sqlalchemy", sqlalchemy.__version__, "|", DATABASE, "|", build_college(engine))


sqlalchemy 2.0.54 | scratch/college.db | {'courses': 10, 'students': 25, 'terms': 4, 'sections': 40, 'enrollments': 228}


## Worked examples

### An async engine for the same college

`create_async_engine` takes a URL like `create_engine`, with an async driver named in it:
`sqlite+aiosqlite` instead of `sqlite`. The helper below is `college_engine` made async. Its event
listens on `async_engine.sync_engine`, the ordinary engine inside the async one, since events belong
to the synchronous core:


In [2]:
def async_college_engine(path, echo=False):
    """An async engine for the college's database file, on the aiosqlite driver, with foreign keys enforced."""
    async_engine = create_async_engine(f"sqlite+aiosqlite:///{path}", echo=echo)

    @event.listens_for(async_engine.sync_engine, "connect")
    def enforce_foreign_keys(dbapi_connection, connection_record):
        cursor = dbapi_connection.cursor()
        cursor.execute("PRAGMA foreign_keys = ON")
        cursor.close()

    return async_engine


async_engine = async_college_engine(DATABASE)
AsyncSessionLocal = async_sessionmaker(async_engine, expire_on_commit=False)


async with AsyncSessionLocal() as session:
    print("foreign keys:", await session.scalar(text("PRAGMA foreign_keys")))
    print("students:    ", await session.scalar(select(func.count()).select_from(Student)))


foreign keys: 1
students:     25


`async with` opened the session and will close it, which is itself a call to the database, and every
query was awaited, at the top level of the cell. A notebook can do that because its kernel runs an
event loop already. Foreign keys are on, and the college is the one Setup built.

### Awaiting every call that reaches the database

The registrar admits Ines Moreau to Biology, with `echo` on:


In [3]:
async_engine.echo = True
async with AsyncSessionLocal() as session:
    ines = Student(name="Ines Moreau", email="imoreau@college.edu", program="Biology", started_on=date(2026, 8, 24))
    session.add(ines)
    await session.commit()
async_engine.echo = False
print(ines, "is student", ines.id)


    BEGIN (implicit)
    INSERT INTO students (name, email, program, started_on) VALUES (?, ?, ?, ?)
    values: ('Ines Moreau', 'imoreau@college.edu', 'Biology', '2026-08-24')
    COMMIT
Student('Ines Moreau', 'Biology') is student 26


`add()` was not awaited: it only hands the object to the session, and sends nothing. `commit()` was,
and it sent the `INSERT` and the `COMMIT`. The session was made with `expire_on_commit=False`, so
`ines.id` could still be read after the commit, and after the session closed, without going back to
the database.

| Awaited, since they can reach the database | Not awaited |
|---|---|
| `execute`, `scalars`, `scalar`, `get`, `stream`, `stream_scalars` | `add`, `add_all`, `expunge` |
| `commit`, `rollback`, `flush`, `refresh`, `delete`, `close` | `select()` and every statement built in Python |

### Results all at once, or streamed

`scalars()` fetches every row before it returns, and `stream_scalars()` fetches them as a loop asks
for them:


In [4]:
FALL_2024_STUDENTS = select(Student.name).where(Student.started_on == date(2024, 8, 26)).order_by(Student.id)

async with AsyncSessionLocal() as session:
    names = (await session.scalars(FALL_2024_STUDENTS)).all()
    print("all at once:", len(names), names[:3])

    streamed = []
    async for name in await session.stream_scalars(FALL_2024_STUDENTS):
        streamed.append(name)
    print("streamed:   ", len(streamed), streamed[:3])


all at once: 9 ['Ana Reyes', 'Daniel Kim', 'Grace Lin']
streamed:    9 ['Ana Reyes', 'Daniel Kim', 'Grace Lin']


`scalars()` had every row in hand once its `await` finished, so `.all()` needed no `await` of its
own. `stream_scalars()` returned a result that fetches as it goes, so the loop over it is an
`async for`, which can wait between rows. Streaming is for results too large to hold at once. For
nine names, either does.

### Relationships: loaded in the query, or awaited

An async program says in the query what it will read, since a lazy load has nowhere to wait:


In [5]:
CHLOE_WITH_COURSES = (
    select(Student)
    .where(Student.id == 3)
    .options(selectinload(Student.enrollments).selectinload(Enrollment.section).selectinload(Section.course))
)

async with AsyncSessionLocal() as session:
    chloe = await session.scalar(CHLOE_WITH_COURSES)
    print([enrollment.section.course.code for enrollment in chloe.enrollments])

    maya = await session.get(Student, 13)
    enrollments = await maya.awaitable_attrs.enrollments
    print(len(enrollments), "enrollments for", maya.name, "loaded when awaited")


['CHE-110', 'CSC-201', 'PSY-101', 'MAT-120', 'ENG-105', 'STA-200']
12 enrollments for Maya Patel loaded when awaited


Every attribute the list read, three relationships deep, was loaded when the query ran, so reading
them needed no database. `awaitable_attrs` is for the attribute a query did not load: awaiting
`maya.awaitable_attrs.enrollments` ran the lazy load where it could wait. It exists because Setup's
`Base` inherits `AsyncAttrs`.

### Synchronous code in an async session: run_sync

A function written for an ordinary `Session`, lazy loads and all, still has a use. `run_sync` calls
it with the synchronous session that the async one wraps, inside the greenlet where waiting is
possible:


In [6]:
def credits_earned(session, student_id):
    """Written for an ordinary Session: every relationship it reads loads lazily."""
    student = session.get(Student, student_id)
    return sum(enrollment.section.course.credits for enrollment in student.enrollments
               if enrollment.grade not in (None, "F"))


async with AsyncSessionLocal() as session:
    print("Chloe Martin has earned", await session.run_sync(credits_earned, 3), "credits")


Chloe Martin has earned 7 credits


Seven credits, as the **Reading Results** notebook worked out in SQL, from a function that never
awaits anything. `run_sync` is how an async program reuses synchronous code, or calls something with
no async version at all, such as `metadata.create_all`, which The idea's first look ran through
`conn.run_sync`.

### Several sessions at once: asyncio.gather

`asyncio.gather` runs several coroutines at once and returns their answers in the order it was given
them. Here is one count for every term, each in a session of its own:


In [7]:
ENROLLMENTS_IN = select(func.count()).select_from(Enrollment).join(Enrollment.section).join(Section.term)


async def enrollments_in(term_name):
    """The number of enrollments in one term, counted in a session of its own."""
    async with AsyncSessionLocal() as session:
        return await session.scalar(ENROLLMENTS_IN.where(Term.name == term_name))


term_names = [name for name, starts in TERMS]
print(dict(zip(term_names, await asyncio.gather(*(enrollments_in(name) for name in term_names)))))


{'Fall 2024': 27, 'Spring 2025': 51, 'Fall 2025': 75, 'Spring 2026': 75}


Four sessions, four connections, and the answers in the order of `term_names`, whichever query
finished first. How much sooner four queries finish this way depends on the database and on how long
each waits for it, which is why the notebook prints no timing.

### Which to use, sync or async

| The program around the database | Use | Why |
|---|---|---|
| a script, a notebook, a batch job, a test | `create_engine` and `Session` | nothing else has to run while it waits |
| a server with `async def` endpoints, such as FastAPI's | `create_async_engine` and `AsyncSession` | while one request waits for the database, the others run |
| async code that must call synchronous ORM code | `await session.run_sync(function)` | the synchronous code runs where it can wait |
| async code that reads relationships | `selectinload` and the other eager loads, or `awaitable_attrs` | an attribute read cannot be awaited |

Synchronous is the default. Reach for async when the program around the database is async already,
and then make every session with `expire_on_commit=False`.

### Transcripts for an advising page, finished

The pieces of this notebook in one function. An advising page shows several students' transcripts
at once, and `transcript` builds one: a session of its own, one query that loads everything the text
will need, and the text written after the session has closed:


In [8]:
TRANSCRIPT = select(Student).options(
    selectinload(Student.enrollments)
    .selectinload(Enrollment.section)
    .options(selectinload(Section.course), selectinload(Section.term))
)


async def transcript(student_id):
    """One student's courses, term by term, read in a session of its own and returned as text."""
    async with AsyncSessionLocal() as session:
        student = await session.scalar(TRANSCRIPT.where(Student.id == student_id))
    lines = [student.name]
    for enrollment in student.enrollments:
        section = enrollment.section
        lines.append(f"    {section.term.name:<12} {section.course.code:<8} {enrollment.grade or 'in progress'}")
    return "\n".join(lines)


for page in await asyncio.gather(*(transcript(student_id) for student_id in (3, 12, 24))):
    print(page)


Chloe Martin
    Fall 2025    CHE-110  C+
    Fall 2025    CSC-201  F
    Fall 2025    PSY-101  B+
    Spring 2026  MAT-120  in progress
    Spring 2026  ENG-105  in progress
    Spring 2026  STA-200  in progress
Liam Murphy
    Fall 2025    BIO-101  D
    Fall 2025    CSC-101  B+
    Fall 2025    HIS-110  C+
    Spring 2026  CHE-110  in progress
    Spring 2026  CSC-201  in progress
    Spring 2026  PSY-101  in progress
Yara Haddad
    Fall 2025    MAT-120  B+
    Fall 2025    ENG-105  C
    Fall 2025    STA-200  F
    Spring 2026  BIO-101  in progress
    Spring 2026  MAT-121  in progress
    Spring 2026  HIS-110  in progress


Three transcripts, gathered at once, and every line written from objects whose session had already
closed. Nothing needed the session any longer: the one query, with its `selectinload`s, had loaded
every attribute the loop read.

### Where each part came from

| In `transcript` | What it relies on | The section that showed it |
|---|---|---|
| `async with AsyncSessionLocal() as session` | a session for every coroutine | Several sessions at once: asyncio.gather |
| `await session.scalar(...)` | awaiting the call that reaches the database | Awaiting every call that reaches the database |
| `selectinload` three relationships deep | relationships loaded in the query | Relationships: loaded in the query, or awaited |
| `await asyncio.gather(...)` | several coroutines at once, their answers in order | Several sessions at once: asyncio.gather |


## Your turn

Six tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than getting
there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlalchemy-deep-dive/16-async-sqlalchemy-solutions.ipynb).

**1.** In an async session, print the names of the History students, in order of id.


In [9]:
# your code here


**2.** Add a course, ART-100, Drawing I, in the Art department, worth 3 credits, with `echo` on, and
print its id after the session has closed.


In [10]:
# your code here


**3.** Stream every enrollment that has a grade with `stream_scalars`, and count the As in an
`async for` loop.


In [11]:
# your code here


**4.** In one query, load section 40 with its enrollments and each enrollment's student, and print
the students' names.


In [12]:
# your code here


**5.** Load the course STA-200 with `session.scalar`, then read its sections through
`awaitable_attrs`, and print their ids.


In [13]:
# your code here


**6.** With `asyncio.gather`, count the students in every program at once, a session for each
program, and print the counts as a dictionary.


In [14]:
# your code here


## Common errors

### sqlalchemy.exc.MissingGreenlet: greenlet_spawn has not been called; can't call await_only() here. Was IO attempted in an unexpected place?


In [15]:
async with AsyncSessionLocal() as session:
    chloe = await session.get(Student, 3)
    print(chloe.name)
    print(len(chloe.enrollments))


Chloe Martin


MissingGreenlet: greenlet_spawn has not been called; can't call await_only() here. Was IO attempted in an unexpected place? (Background on this error at: https://sqlalche.me/e/20/xd2s)

`session.get` loaded Chloe Martin's columns, and the name printed. `enrollments` was never loaded,
and reading it would have been a lazy load, a query from inside an attribute read, where nothing can
await it. SQLAlchemy raised rather than hold the event loop. The same error comes from an attribute a
commit expired, in a session left at the default `expire_on_commit=True`:


In [16]:
Expiring = async_sessionmaker(async_engine)                         # expire_on_commit left at True

async with Expiring() as session:
    ben = await session.get(Student, 2)
    await session.commit()                                          # expires every object in the session
    print(ben.name)


MissingGreenlet: greenlet_spawn has not been called; can't call await_only() here. Was IO attempted in an unexpected place? (Background on this error at: https://sqlalche.me/e/20/xd2s)

Load what the code will read in the query, and refresh what a commit expired, both of them awaited:


In [17]:
async with AsyncSessionLocal() as session:
    chloe = await session.get(Student, 3, options=[selectinload(Student.enrollments)])
    print(chloe.name, "has", len(chloe.enrollments), "enrollments")

async with Expiring() as session:
    ben = await session.get(Student, 2)
    await session.commit()
    await session.refresh(ben)
    print(ben.name, "after the refresh")


Chloe Martin has 6 enrollments
Ben Okafor after the refresh


### AttributeError: 'coroutine' object has no attribute 'all'


In [18]:
async with AsyncSessionLocal() as session:
    pending = session.scalars(select(Student.name).order_by(Student.id))    # the await left out
    print(type(pending).__name__)
    try:
        pending.all()
    except AttributeError as error:
        print("AttributeError:", error)
    pending.close()


coroutine
AttributeError: 'coroutine' object has no attribute 'all'


Without `await`, `session.scalars(...)` returned a coroutine, the query not yet sent, and a
coroutine has no `all()`. The cell closes it, which marks a coroutine that will never run as
finished; one dropped unfinished makes Python warn that it was never awaited, a warning this notebook
does not print, since it names a temporary file. Await the call:


In [19]:
async with AsyncSessionLocal() as session:
    student_names = (await session.scalars(select(Student.name).order_by(Student.id))).all()
print(len(student_names), "students, the first", student_names[0])


26 students, the first Ana Reyes


### RuntimeError: asyncio.run() cannot be called from a running event loop


In [20]:
async def count_students():
    async with AsyncSessionLocal() as session:
        return await session.scalar(select(func.count()).select_from(Student))


counting = count_students()
try:
    print(asyncio.run(counting))
except RuntimeError as error:
    print("RuntimeError:", error)
    counting.close()


RuntimeError: asyncio.run() cannot be called from a running event loop


The notebook's kernel runs an event loop already, which is what lets a cell `await` at its top
level, and `asyncio.run()` starts a loop of its own, which it cannot do inside another. The
coroutine never ran, and the cell closes it for the reason the last error gave. In a script, where no
loop runs until something starts one, `asyncio.run(main())` is the way in, as The idea's first look
shows. In a notebook, await:


In [21]:
print(await count_students(), "students")


26 students


### sqlalchemy.exc.InvalidRequestError: This session is provisioning a new connection; concurrent operations are not permitted


In [22]:
async def enrollments_in_shared(session, term_name):
    return await session.scalar(ENROLLMENTS_IN.where(Term.name == term_name))


starting_engine = async_college_engine(DATABASE)                    # no connections yet, like a program just started
async with async_sessionmaker(starting_engine)() as session:
    results = await asyncio.gather(*(enrollments_in_shared(session, name) for name in term_names),
                                   return_exceptions=True)
await starting_engine.dispose()
for name, result in zip(term_names, results):
    print(f"{name:<12}", result if isinstance(result, int) else f"{type(result).__name__}: {result}")


Fall 2024    27
Spring 2025  InvalidRequestError: This session is provisioning a new connection; concurrent operations are not permitted (Background on this error at: https://sqlalche.me/e/20/isce)
Fall 2025    InvalidRequestError: This session is provisioning a new connection; concurrent operations are not permitted (Background on this error at: https://sqlalche.me/e/20/isce)
Spring 2026  InvalidRequestError: This session is provisioning a new connection; concurrent operations are not permitted (Background on this error at: https://sqlalche.me/e/20/isce)


The four coroutines shared one session, on an engine with no connections yet, as at the start of a
program. The first began to open a connection, and the other three found the session in the middle
of that and were refused. `return_exceptions=True` made `gather` wait for all four and hand the
errors back as values, so that the session closed only after every coroutine was done with it. On
the notebook's own engine, whose connections are open already, the same sharing goes unnoticed:


In [23]:
async with AsyncSessionLocal() as session:
    results = await asyncio.gather(*(enrollments_in_shared(session, name) for name in term_names),
                                   return_exceptions=True)
print(results)


[27, 51, 75, 75]


No error this time: the session had its connection at once, and SQLite's driver ran the four
statements on it one after another. The code is the same, and whether it fails depends on the state
of the pool, which makes it a bug that shows on some runs and not others. A session sends one
statement at a time, so coroutines that run at once each need their own, as in `enrollments_in`:


In [24]:
print(dict(zip(term_names, await asyncio.gather(*(enrollments_in(name) for name in term_names)))))


{'Fall 2024': 27, 'Spring 2025': 51, 'Fall 2025': 75, 'Spring 2026': 75}


Last, both engines let go of the file, the async one with an `await`, and this cell removes the
scratch folder, with the database in it:


In [25]:
await async_engine.dispose()
engine.dispose()
shutil.rmtree("scratch")

print("scratch still there:", Path("scratch").exists())


scratch still there: False


## Recap

- `create_async_engine` with an async driver, `sqlite+aiosqlite` here, and `async_sessionmaker` make
  async sessions, whose calls to the database are all awaited, and `add()` is not.
- An attribute read cannot be awaited, so an async program loads relationships in the query, or
  awaits `awaitable_attrs`, and makes its sessions with `expire_on_commit=False`.
- `run_sync` runs synchronous ORM code, lazy loads included, inside an async session.
- `asyncio.gather` runs several coroutines at once, each with a session of its own.
- A notebook awaits at the top level of a cell, and a script starts its event loop with
  `asyncio.run()`.


## What is next

The **Migrations with Alembic** notebook changes the college's tables once they hold data:
`alembic revision --autogenerate`, `upgrade` and `downgrade`, and the batch mode SQLite needs to
alter a table.


---

&#8592; **Previous:** [Cascades and Deletes](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlalchemy-deep-dive/15-cascades-and-deletes.ipynb)  &nbsp;·&nbsp;  [SQLAlchemy, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlalchemy-deep-dive.html)  &nbsp;·&nbsp;  **Next:** [Migrations with Alembic](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlalchemy-deep-dive/17-migrations-with-alembic.ipynb) &#8594;
